# 05 - Data Writing
Persist gold Delta tables from the enriched silver table, optimize them, and create a BI-facing view.

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg, round as spark_round

In [ ]:
enriched = spark.read.table("ola_lakehouse.silver.trips_enriched")

### Recompute and write the four gold aggregates

In [ ]:
daily_revenue = enriched.filter(col("status")=="COMPLETED") \
    .groupBy("trip_date","trip_city") \
    .agg(spark_sum("fare_amount").alias("total_revenue"), count("trip_id").alias("completed_trips"))

vehicle_perf = enriched.filter(col("status")=="COMPLETED") \
    .groupBy("vehicle_type") \
    .agg(count("trip_id").alias("total_trips"), spark_sum("fare_amount").alias("total_revenue"))

leaderboard = enriched.filter(col("status")=="COMPLETED") \
    .groupBy("driver_id","driver_name") \
    .agg(count("trip_id").alias("completed_trips"), spark_sum("fare_amount").alias("total_revenue"))

cancellation_rate = enriched.groupBy("trip_city") \
    .agg(
        count("trip_id").alias("total_trips"),
        spark_sum((col("status")=="CANCELLED").cast("int")).alias("cancelled_trips")
    ).withColumn("cancellation_rate_pct", spark_round(col("cancelled_trips")/col("total_trips")*100, 2))

In [ ]:
daily_revenue.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.gold.daily_revenue_by_city")
vehicle_perf.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.gold.vehicle_type_performance")
leaderboard.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.gold.driver_leaderboard")
cancellation_rate.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.gold.cancellation_rate_by_city")

print("Gold tables written.")

### Optimize gold tables

In [ ]:
%sql
OPTIMIZE ola_lakehouse.gold.daily_revenue_by_city ZORDER BY (trip_city);
OPTIMIZE ola_lakehouse.gold.driver_leaderboard;

### BI-facing view

In [ ]:
%sql
CREATE OR REPLACE VIEW ola_lakehouse.gold.vw_city_performance AS
SELECT
  r.trip_city,
  SUM(r.total_revenue) AS total_revenue,
  SUM(r.completed_trips) AS total_completed_trips,
  c.cancellation_rate_pct
FROM ola_lakehouse.gold.daily_revenue_by_city r
JOIN ola_lakehouse.gold.cancellation_rate_by_city c
  ON r.trip_city = c.trip_city
GROUP BY r.trip_city, c.cancellation_rate_pct
ORDER BY total_revenue DESC

In [ ]:
%sql
SELECT * FROM ola_lakehouse.gold.vw_city_performance